In [2]:

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, accuracy_score

dataset = pd.read_csv('PhiUSIIL_Phishing_URL_Dataset.csv')

columns_to_drop = [
    'Domain', 'TLD', 'Title', 'URL',
    'URLSimilarityIndex',
    'TLDLegitimateProb', 'URLCharProb'
]

numeric_cols = dataset.columns[2:-1].difference(columns_to_drop)

X = dataset[list(numeric_cols)]
y = dataset['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), list(numeric_cols))
])

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

classifier = GaussianNB()
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

print("--- Test Set Results ---")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nAccuracy Score: {:.2f} %".format(accuracy_score(y_test, y_pred) * 100))

print("\n--- 10-Fold Cross-Validation ---")
accuracies = cross_val_score(estimator=classifier, X=X_train, y=y_train, cv=10)
print("Mean Accuracy: {:.2f} %".format(accuracies.mean() * 100))
print("Standard Deviation: {:.2f} %".format(accuracies.std() * 100))

print(dataset["label"].value_counts())
url = "https://www.southbankmosaics.com"

print(dataset[dataset["URL"] == url][["URL", "label"]])

--- Test Set Results ---
Confusion Matrix:
[[17210  2914]
 [   12 27023]]

Accuracy Score: 93.80 %

--- 10-Fold Cross-Validation ---
Mean Accuracy: 93.78 %
Standard Deviation: 0.14 %
label
1    134850
0    100945
Name: count, dtype: int64
                                URL  label
0  https://www.southbankmosaics.com      1


Data leakage problems. URLSimilarityIndex column is a dead giveaway to the model when the email is going to be a phish or not, model is only looking for that one pattern, will exclude along with a few other columns to try and get the accuracy to a more realistic number. Final model comparison testing. TF-IDF for the non-numerical columns also seems to ineffective, resulting in little to no changes in accuracy.

Setup	Test Accuracy
All numeric features	99.97%
Minus URLSimilarityIndex only	93.77%
Minus URLSimilarityIndex, TLDLegitimateProb, URLCharProb	93.80%
Minus those 3 + 6 more "aggressive" drops	92.38%
URL text only (TF-IDF)	96.20%